In [1]:
import os, sys

if not os.path.isdir("Explanations"):
    os.chdir("..")

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print("cwd:", os.getcwd())
print("Explanations/ exists:", os.path.isdir("Explanations"))


cwd: c:\Users\Offic\OneDrive\Desktop\Projects\5 - XAI Evaluation Paper
Explanations/ exists: True


In [2]:
from src.evaluation import create_experiment
from src.fidelity import run_fidelity
from src.utils import Logger
import os

os.makedirs("Evaluation/logs", exist_ok=True)
# logger = Logger("Evaluation/logs", filename="setup_log.txt")
# experiment_id = create_experiment("Evaluation", {"purpose": "fidelity run"

DATASETS = [
    ("healthcare", "pima_diabetes"),
    ("healthcare", "breast_cancer_wisconsin"),
    ("healthcare", "heart_disease_uci"),
    ("finance", "loan_default"),
    ("finance", "financial_distress"),
    ("finance", "credit_card_fraud_2023"),
]
MODEL_NAMES = ["rf", "xgb"]

logger = Logger("Evaluation/logs", filename="ledger.log")
experiment_id = create_experiment(
    "Evaluation",
    {"phase": "fidelity", "datasets": DATASETS, "models": MODEL_NAMES},
    logger,
)
print("experiment_id:", experiment_id)

Created experiment_id=exp_20260716T042631_9a2e7741, logged to Evaluation\run_ledger.csv
experiment_id: exp_20260716T042631_9a2e7741


In [3]:
import os
from src.fidelity import run_fidelity

results = {}
for domain, dataset_name in DATASETS:
    for model_name in MODEL_NAMES:
        run_dir = os.path.join("Explanations", domain, dataset_name, model_name)
        if not os.path.isdir(run_dir):
            print(f"SKIP {domain}/{dataset_name}/{model_name}: no Explanations/ artifacts yet")
            continue
        try:
            rows = run_fidelity({
                "domain": domain,
                "dataset_name": dataset_name,
                "model_name": model_name,
                "experiment_id": experiment_id,
            })
            results[(domain, dataset_name, model_name)] = len(rows)
        except Exception as e:
            print(f"FAILED {domain}/{dataset_name}/{model_name}: {e}")

print()
print(f"{len(results)}/{len(DATASETS) * len(MODEL_NAMES)} combos completed.")
results


------------------------------------------------------------
FIDELITY: pima_diabetes x rf
------------------------------------------------------------
Loaded model (skops): Models\healthcare\pima_diabetes\rf\model\model.skops
Feature-group map for pima_diabetes: 8 original features (from 8 model-input columns).
Baseline vector computed for healthcare/pima_diabetes from X_train_prebalance.csv (614 rows): 0 categorical, 8 numerical.
Wrote 1232 rows to Fidelity\healthcare\pima_diabetes\rf\shap\masked_predictions.csv
Wrote 1232 rows to Fidelity\healthcare\pima_diabetes\rf\lime\masked_predictions.csv
Appended 3080 row(s) to Evaluation\metrics_long.csv

------------------------------------------------------------
DONE
------------------------------------------------------------
Fidelity: 3080 metrics_long rows written across ['shap', 'lime'].

------------------------------------------------------------
FIDELITY: pima_diabetes x xgb
----------------------------------------------------------

{('healthcare', 'pima_diabetes', 'rf'): 3080,
 ('healthcare', 'pima_diabetes', 'xgb'): 3080,
 ('healthcare', 'breast_cancer_wisconsin', 'rf'): 2280,
 ('healthcare', 'breast_cancer_wisconsin', 'xgb'): 2280,
 ('healthcare', 'heart_disease_uci', 'rf'): 3680,
 ('healthcare', 'heart_disease_uci', 'xgb'): 3680,
 ('finance', 'loan_default', 'rf'): 10000,
 ('finance', 'loan_default', 'xgb'): 10000,
 ('finance', 'financial_distress', 'rf'): 14680,
 ('finance', 'financial_distress', 'xgb'): 14680,
 ('finance', 'credit_card_fraud_2023', 'rf'): 10000,
 ('finance', 'credit_card_fraud_2023', 'xgb'): 10000}

In [ ]:

# from src.evaluation import create_experiment
# from src.fidelity import run_fidelity
# from src.utils import Logger
# import os

# os.makedirs("Evaluation/logs", exist_ok=True)
# logger = Logger("Evaluation/logs", filename="setup_log.txt")
# experiment_id = create_experiment("Evaluation", {"purpose": "fidelity run"}, logger)

# # repeat per dataset x model as artifacts become available
# run_fidelity({
#     "domain": "healthcare",
#     "dataset_name": "pima_diabetes",
#     "model_name": "rf",
#     "experiment_id": experiment_id,
# })

In [ ]:
# combos = [
#     ("healthcare", "pima_diabetes", "rf"),
#     ("healthcare", "pima_diabetes", "xgb"),
#     ("finance", "loan_default", "rf"),
#     ("finance", "loan_default", "xgb"),
#     # ... as the rest come online
# ]
# for domain, dataset_name, model_name in combos:
#     run_fidelity({
#         "domain": domain, "dataset_name": dataset_name, "model_name": model_name,
#         "experiment_id": experiment_id,
#     })

In [4]:
import pandas as pd
df = pd.read_csv("Evaluation/metrics_long.csv")
df[df.experiment_id == experiment_id].groupby(["dataset", "model", "explainer", "metric_name"])["value"].mean().unstack("metric_name")

metric_name                              aopc_comprehensiveness  \
dataset                 model explainer                           
breast_cancer_wisconsin rf    lime                     0.192699   
                              shap                     0.214107   
                        xgb   lime                     0.218615   
                              shap                     0.250404   
credit_card_fraud_2023  rf    lime                     0.424172   
                              shap                     0.465817   
                        xgb   lime                     0.465680   
                              shap                     0.571288   
financial_distress      rf    lime                     0.039028   
                              shap                     0.044181   
                        xgb   lime                     0.029183   
                              shap                     0.033402   
heart_disease_uci       rf    lime                     0.137180   
                              shap                     0.196424   
                        xgb   lime                     0.130221   
                              shap                     0.223846   
loan_default            rf    lime                     0.050162   
                              shap                     0.066184   
                        xgb   lime                     0.010860   
                              shap                     0.035340   
pima_diabetes           rf    lime                     0.181600   
                              shap                     0.200826   
                        xgb   lime                     0.162463   
                              shap                     0.173508   

metric_name                              aopc_sufficiency  comprehensiveness  \
dataset                 model explainer                                        
breast_cancer_wisconsin rf    lime               0.063940           0.192699   
                              shap               0.046560           0.214107   
                        xgb   lime               0.089431           0.218615   
                              shap               0.067842           0.250404   
credit_card_fraud_2023  rf    lime               0.084356           0.424172   
                              shap               0.064018           0.465817   
                        xgb   lime               0.049490           0.465680   
                              shap               0.014869           0.571288   
financial_distress      rf    lime               0.004358           0.039028   
                              shap              -0.002279           0.044181   
                        xgb   lime               0.003427           0.029183   
                              shap               0.005924           0.033402   
heart_disease_uci       rf    lime               0.069348           0.137180   
                              shap               0.017940           0.196424   
                        xgb   lime               0.128803           0.130221   
                              shap               0.053853           0.223846   
loan_default            rf    lime              -0.041279           0.050162   
                              shap              -0.050774           0.066184   
                        xgb   lime              -0.031742           0.010860   
                              shap              -0.044387           0.035340   
pima_diabetes           rf    lime               0.025528           0.181600   
                              shap               0.011987           0.200826   
                        xgb   lime               0.020442           0.162463   
                              shap               0.009910           0.173508   

metric_name                              sufficiency  
dataset                 model explainer               
breast_cancer_wisconsin rf    lime          0.063940  
           

In [5]:
# aggregated view only (recommended default) -- one row per instance already
agg_view = df[(df.experiment_id == experiment_id) & (df.metric_name.str.startswith("aopc_") | ~df.metric_property.eq("Fidelity"))]
agg_view.groupby(["dataset", "model", "explainer", "metric_name"])["value"].mean().unstack("metric_name")

metric_name                              aopc_comprehensiveness  \
dataset                 model explainer                           
breast_cancer_wisconsin rf    lime                     0.192699   
                              shap                     0.214107   
                        xgb   lime                     0.218615   
                              shap                     0.250404   
credit_card_fraud_2023  rf    lime                     0.424172   
                              shap                     0.465817   
                        xgb   lime                     0.465680   
                              shap                     0.571288   
financial_distress      rf    lime                     0.039028   
                              shap                     0.044181   
                        xgb   lime                     0.029183   
                              shap                     0.033402   
heart_disease_uci       rf    lime                     0.137180   
                              shap                     0.196424   
                        xgb   lime                     0.130221   
                              shap                     0.223846   
loan_default            rf    lime                     0.050162   
                              shap                     0.066184   
                        xgb   lime                     0.010860   
                              shap                     0.035340   
pima_diabetes           rf    lime                     0.181600   
                              shap                     0.200826   
                        xgb   lime                     0.162463   
                              shap                     0.173508   

metric_name                              aopc_sufficiency  
dataset                 model explainer                    
breast_cancer_wisconsin rf    lime               0.063940  
                              shap               0.046560  
                        xgb   lime               0.089431  
                              shap               0.067842  
credit_card_fraud_2023  rf    lime               0.084356  
                              shap               0.064018  
                        xgb   lime               0.049490  
                              shap               0.014869  
financial_distress      rf    lime               0.004358  
                              shap              -0.002279  
                        xgb   lime               0.003427  
                              shap               0.005924  
heart_disease_uci       rf    lime               0.069348  
                              shap               0.017940  
                        xgb   lime               0.128803  
                              shap               0.053853  
loan_default            rf    lime              -0.041279  
                              shap              -0.050774  
                        xgb   lime              -0.031742  
                              shap              -0.044387  
pima_diabetes           rf    lime               0.025528  
                              shap               0.011987  
                        xgb   lime               0.020442  
                              shap               0.009910

In [6]:
# per-k breakdown, if you want to see how comprehensiveness/sufficiency scale with mask_fraction
fid_by_k = df[(df.experiment_id == experiment_id) & df.mask_fraction.notna()]
fid_by_k.groupby(["dataset", "model", "explainer", "metric_name", "mask_fraction"])["value"].mean().unstack("metric_name")

metric_name                                            comprehensiveness  \
dataset                 model explainer mask_fraction                      
breast_cancer_wisconsin rf    lime      0.1                     0.114751   
                                        0.2                     0.173768   
                                        0.3                     0.213212   
                                        0.5                     0.269066   
                              shap      0.1                     0.132534   
...                                                                  ...   
pima_diabetes           xgb   lime      0.5                     0.198207   
                              shap      0.1                     0.140793   
                                        0.2                     0.173901   
                                        0.3                     0.173901   
                                        0.5                     0.205437   

metric_name                                            sufficiency  
dataset                 model explainer mask_fraction               
breast_cancer_wisconsin rf    lime      0.1               0.123762  
                                        0.2               0.072920  
                                        0.3               0.048753  
                                        0.5               0.010323  
                              shap      0.1               0.107829  
...                                                            ...  
pima_diabetes           xgb   lime      0.5               0.006405  
                              shap      0.1               0.020775  
                                        0.2               0.007413  
                                        0.3               0.007413  
                                        0.5               0.004040  

[96 rows x 2 columns]